# Language-model prompting benchmark

Runs an open-weights instruction model with the versioned few-shot prompt on every test sentence, saves the raw answers, and scores extraction and end-to-end results with the same evaluators as the BERT extractor. Answers that are not valid JSON, and aspects that are not in the review or carry an unknown label, are counted as invalid. No API key is used: the model runs on the Colab GPU, so cost is not applicable and latency is measured.

## Setup

Run this on a Colab GPU runtime (*Runtime → Change runtime type → GPU*). The cell clones the repository, installs the dependencies and checks the environment. Colab keeps its own `torch`, so it is left as it is.

In [1]:
import os

if not os.path.isdir("aspect-based-sentiment-analysis"):
    !git clone -q https://github.com/Shivam-Shukla0/aspect-based-sentiment-analysis.git
%cd aspect-based-sentiment-analysis
!pip install -q -r requirements.txt
!pip install -q -e . --no-deps
!python -m scripts.verify_environment --require-gpu

/content/run/aspect-based-sentiment-analysis


  Installing build dependencies ... done


  Checking if build backend supports build_editable ... done


  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done


  Building editable for absa (pyproject.toml) ... done


python          3.13.16
numpy           2.4.6
pandas          2.2.3
scikit-learn    1.6.1
lxml            6.1.3
torch           2.14.1+cu130
transformers    5.22.1
accelerate      1.15.0
sentencepiece   0.2.2
matplotlib      3.10.0
flask           3.1.3
absa            0.1.0
device          NVIDIA L4 (CUDA 13.0, 1 device)
environment ok


## Data

The official SemEval-2014 Task 4 files are distributed through the ILSP Metashare repository and need an account and acceptance of its licence (see `data/README.md`). This cell uses those files when they are in Google Drive (set `USE_DRIVE = True` after mounting Drive), and otherwise downloads the same four files from a public Hugging Face mirror of the dataset. The mirror did not come through the official route, so the file checksums are printed and the report states this limitation. Nothing in `data/` is committed to Git.

In [2]:
from pathlib import Path

USE_DRIVE = False  # True: copy the official files from DRIVE_DIR (mount Drive first)
DRIVE_DIR = Path("/content/drive/MyDrive/absa/semeval2014")
MIRROR = "https://huggingface.co/datasets/alexcadillon/SemEval2014Task4/resolve/main"
TRAIN = "SemEval%2714-ABSA-TrainData_v2%20%26%20AnnotationGuidelines"
FILES = {
    "Laptop_Train_v2.xml": (f"{MIRROR}/{TRAIN}/Laptop_Train_v2.xml", "laptop", "train"),
    "Restaurants_Train_v2.xml": (
        f"{MIRROR}/{TRAIN}/Restaurants_Train_v2.xml",
        "restaurant",
        "train",
    ),
    "Laptops_Test_Gold.xml": (
        f"{MIRROR}/ABSA_Gold_TestData/Laptops_Test_Gold.xml",
        "laptop",
        "test",
    ),
    "Restaurants_Test_Gold.xml": (
        f"{MIRROR}/ABSA_Gold_TestData/Restaurants_Test_Gold.xml",
        "restaurant",
        "test",
    ),
}
Path("data/raw").mkdir(parents=True, exist_ok=True)
for name, (url, domain, split) in FILES.items():
    if USE_DRIVE:
        !cp "{DRIVE_DIR / name}" data/raw/
    else:
        !curl -sL -o data/raw/{name} "{url}"
    !python -m scripts.build_processed data/raw/{name} --domain {domain} --split {split} > /dev/null
!sha256sum data/raw/*.xml
!wc -l data/processed/*.jsonl

aa3a30c9dd7e0eacf9b69e898eb4118a36c1b4f391a30a08dfc41a38d8b8745b  data/raw/Laptops_Test_Gold.xml
efec2ed73b4efb83b47c697114d800149c558118884afa80362a6e975ba7248c  data/raw/Laptop_Train_v2.xml
f21509cfa37e16534cd5b2da043be487355b64ef48fe8d6aaacaeca6b49cc0fb  data/raw/Restaurants_Test_Gold.xml
223601da1bded6caa4ef9cf91a7007578141ca6d8ed50d5a5c217565f89d2fc5  data/raw/Restaurants_Train_v2.xml


    654 data/processed/laptop_test.jsonl
    378 data/processed/laptop_test_no_aspect.jsonl
   2358 data/processed/laptop_train.jsonl
   1557 data/processed/laptop_train_no_aspect.jsonl
   1134 data/processed/restaurant_test.jsonl
    194 data/processed/restaurant_test_no_aspect.jsonl
   3693 data/processed/restaurant_train.jsonl
   1020 data/processed/restaurant_train_no_aspect.jsonl
  10988 total


## Run the benchmark

Decoding is greedy, so a rerun gives the same answers. The raw answers are written to `outputs/llm_<domain>/responses.jsonl`; `--from-responses` recomputes the scores from that file without the model.

In [3]:
MODEL = "Qwen/Qwen2.5-7B-Instruct"
for domain in ("laptop", "restaurant"):
    !python -m scripts.benchmark_llm --domain {domain} --model-name {MODEL} --batch-size 32 2>&1 | grep -E "^{domain}"

laptop: extraction F1 0.4928, end-to-end F1 0.4108, unparsable 0


restaurant: extraction F1 0.6992, end-to-end F1 0.6235, unparsable 0


## Results

In [4]:
import json

import pandas as pd

rows = []
for domain in ("laptop", "restaurant"):
    summary = json.load(open(f"outputs/llm_{domain}/summary.json"))
    for name, key in (
        ("extraction, exact match", "extraction_exact"),
        ("extraction, overlap match", "extraction_overlap"),
        ("end to end", "end_to_end"),
    ):
        rows.append(
            {
                "domain": domain,
                "setting": name,
                **{k: round(summary[key][k], 3) for k in ("precision", "recall", "f1")},
            }
        )
    print(
        domain,
        f"- {summary['model']}, prompt {summary['prompt']} {summary['prompt_version']}, {summary['sentences']} sentences",
    )
    print(
        f"   unparsable answers: {summary['unparsable_answers']}, invalid aspects: {summary['invalid_aspects']}, "
        f"mean latency {summary['mean_latency_seconds']:.3f} s per sentence, cost: {summary['cost']}"
    )
pd.DataFrame(rows)

laptop - Qwen/Qwen2.5-7B-Instruct, prompt few-shot v1.0, 800 sentences
   unparsable answers: 0, invalid aspects: 29, mean latency 0.435 s per sentence, cost: not applicable (open-weights model run on a Colab GPU)
restaurant - Qwen/Qwen2.5-7B-Instruct, prompt few-shot v1.0, 800 sentences
   unparsable answers: 0, invalid aspects: 34, mean latency 0.473 s per sentence, cost: not applicable (open-weights model run on a Colab GPU)


,domain,setting,precision,recall,f1
0,laptop,"extraction, exact match",0.430,0.576,0.493
1,laptop,"extraction, overlap match",0.539,0.722,0.617
2,laptop,end to end,0.355,0.487,0.411
3,restaurant,"extraction, exact match",0.709,0.690,0.699
4,restaurant,"extraction, overlap match",0.784,0.763,0.773
5,restaurant,end to end,0.628,0.619,0.623


## Raw answers and errors

In [5]:
for domain in ("laptop", "restaurant"):
    print(domain, "- first raw answers:")
    with open(f"outputs/llm_{domain}/responses.jsonl", encoding="utf-8") as handle:
        for _, line in zip(range(3), handle):
            record = json.loads(line)
            print("  review:", record["review"][:80])
            print("  answer:", record["raw"][:160].replace("\n", " "))
    report = json.load(open(f"outputs/llm_{domain}/test_report.json"))
    print("  error counts:", report["error_counts"])

laptop - first raw answers:
  review: Boot time is super fast, around anywhere from 35 seconds to 1 minute.
  answer: [{"aspect": "boot time", "sentiment": "positive"}]
  review: tech support would not fix the problem unless I bought your plan for $150 plus.
  answer: [{"aspect": "tech support", "sentiment": "negative"}]
  review: Set up was easy.
  answer: [{"aspect": "set up", "sentiment": "positive"}]
  error counts: {'boundary': 95, 'missed': 182, 'spurious': 404}
restaurant - first raw answers:
  review: The bread is top notch as well.
  answer: [{"aspect": "bread", "sentiment": "positive"}]
  review: I have to say they have one of the fastest delivery times in the city.
  answer: [{"aspect": "delivery times", "sentiment": "positive"}]
  review: Food is always fresh and hot- ready to eat!
  answer: [{"aspect": "food", "sentiment": "positive"}]
  error counts: {'boundary': 83, 'missed': 269, 'spurious': 238}
